# Single-turn coding control (Qwen3-32B) — DebugBench vs. the frozen value axis

**Preliminary trust-building check**, not the paper headline number — DebugBench
Python problems (random seeded subset of 150, not the full 225), frozen
value axis, layer 49, thinking ON. Isolates whether the Stage-2 agentic
transfer null (`stage2/`) is a *domain* effect (ICRL chat -> code) or a
*horizon* effect (single-turn -> multi-step agentic): here the model never
generates anything, it's prefilled with a given correct/corrupted solution
and we read activations while it processes the code.

**Runtime:** Colab **A100 80GB** (bf16 32B). One GPU — do not set `CUDA_VISIBLE_DEVICES`.

**v2 update:** adds `proj_window_mean`, the anchor paper's actual metric
(“the average value-axis projection on the assistant tokens after the
bug”, og_paper.tex) — a diff-anchored window between original and
corrupted code, not the whole-span mean or bare final token, both of
which turned out to be confounded by a positional ramp unrelated to
correctness. Output filename changed to `projections_v2.parquet`
specifically so this run can't be silently skipped by the old v1 file's
resume checkpoint (schema differs; resume only checks `slug`, which
exists in both, so reusing the old filename would look “already done”).

**Outputs (on Drive):**
- `problems_manifest.json` (prepared DebugBench problems + corruptions, unchanged from v1)
- `projections_v2.parquet` (per-layer projections, resumable checkpoint)
- `report/report.json`, `report/layer_sweep.png`

**Requirements:** `value_axis_32b.npy` + `axis_manifest_32b.json` from Stage 1
(same two files `project_full_32b_colab.ipynb` uses). DebugBench itself is a
public Hugging Face dataset — no upload needed for it.

**ETA:** prep is minutes (CPU + network); projection is GPU-bound but small
(150 problems x ~2-3 forward passes each) — much shorter than the Stage-2
agentic run. Resume is built in — re-run the project cell.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Need GPU (A100 80GB for Qwen3-32B bf16)'
name = torch.cuda.get_device_name(0)
vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
print(name, 'VRAM GB:', vram)
assert vram >= 70, (
    f'Got {vram} GB — need A100 80GB / high-RAM. '
    'Runtime -> Change runtime type -> A100.'
)

In [ ]:
import os
REPO = '/content/failure_prediction_research'
if not os.path.isdir(REPO):
    !git clone https://github.com/abdelmagid07/failure_prediction_research.git {REPO}
else:
    %cd {REPO}
    !git pull
%cd {REPO}
!pip install -q -e stage1 -e stage2 -e single_turn_control

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/failure_prediction_research/single_turn_control_32b')
OUT_DIR = DRIVE_ROOT / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('DRIVE_ROOT:', DRIVE_ROOT)
print('OUT_DIR:', OUT_DIR)

In [ ]:
PRIMARY_LAYER = 49          # informational; analyze.py reads it from axis_manifest_32b.json
MODEL = 'Qwen/Qwen3-32B'
N_LAYERS = 64
N_PROBLEMS = 150            # preliminary check, not the paper's full 225
SEED = 42

print('PRIMARY_LAYER', PRIMARY_LAYER)
print('MODEL', MODEL)
print('N_PROBLEMS', N_PROBLEMS)

## Upload inputs (two dialogs)

1. `value_axis_32b.npy`
2. `axis_manifest_32b.json`

(Same two files as `stage2/notebooks/project_full_32b_colab.ipynb` — the
frozen Stage-1 axis is never rebuilt here.)

In [ ]:
import json, shutil
from pathlib import Path
from google.colab import files
import numpy as np

REPO = Path('/content/failure_prediction_research')
AXIS_DIR = REPO / 'stage1' / 'data'
AXIS_DIR.mkdir(parents=True, exist_ok=True)

def _take_one(upload_dict, dest: Path, *, expect_suffix: str | None = None):
    assert len(upload_dict) == 1, f'Upload exactly one file, got {list(upload_dict)}'
    name = next(iter(upload_dict))
    src = Path(name)
    if expect_suffix is not None:
        assert src.suffix.lower() == expect_suffix.lower(), (
            f'Expected {expect_suffix}, got {src.name}'
        )
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        dest.unlink()
    shutil.move(str(src), dest)
    print('saved ->', dest)
    return dest

print('1/2 — upload value_axis_32b.npy')
AXIS = _take_one(files.upload(), AXIS_DIR / 'value_axis_32b.npy', expect_suffix='.npy')
axis = np.load(AXIS)
print('axis shape', axis.shape, '(expect 64 x 5120)')
assert axis.shape == (64, 5120), axis.shape

print('2/2 — upload axis_manifest_32b.json')
MANIFEST = _take_one(
    files.upload(), AXIS_DIR / 'axis_manifest_32b.json', expect_suffix='.json'
)
manifest = json.loads(MANIFEST.read_text())
print(
    'manifest primary_layer=', manifest.get('primary_layer'),
    'primary_auroc=', manifest.get('primary_auroc'),
    'enable_thinking=', manifest.get('enable_thinking'),
)

## Prepare (CPU) — DebugBench -> corrupted variants -> rendered prompts

Public dataset, no upload needed. Writes `problems_manifest.json` straight to
Drive (small + fast; no local/mirror split needed for this step).

In [ ]:
import subprocess, sys
from pathlib import Path

REPO = Path('/content/failure_prediction_research')
MANIFEST_PATH = OUT_DIR / 'problems_manifest.json'

cmd = [
    sys.executable, '-u', '-m', 'single_turn_control.prepare',
    '--n-problems', str(N_PROBLEMS),
    '--seed', str(SEED),
    '--model', MODEL,
    '--enable-thinking',
    '--output', str(MANIFEST_PATH),
]
print('CMD:', ' '.join(cmd), flush=True)
proc = subprocess.Popen(
    cmd, cwd=str(REPO / 'single_turn_control'),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
rc = proc.wait()
print('prepare exit:', rc, flush=True)
assert rc == 0 and MANIFEST_PATH.exists(), MANIFEST_PATH
print('problems_manifest ->', MANIFEST_PATH)

## Project (GPU)

All layers, one forward pass per (problem, variant). Checkpoints after **each
problem** on local disk, mirrored to Drive every 10 problems. Re-run to
**resume** — a slug already present in the output parquet is skipped.

In [ ]:
import subprocess, sys, shutil
from pathlib import Path

REPO = Path('/content/failure_prediction_research')
AXIS = REPO / 'stage1' / 'data' / 'value_axis_32b.npy'

# v2: schema changed (variant/side pairing + proj_window_mean, the anchor
# paper's actual diff-window readout). MUST use a new filename here -- the
# resume logic only checks the 'slug' column, which exists in both schemas,
# so pointing at the old file would make it think every problem is already
# done and silently skip all of them instead of recomputing under v2.
LOCAL_OUT = Path('/content/single_turn_control_ckpt')
LOCAL_OUT.mkdir(parents=True, exist_ok=True)
PROJ = LOCAL_OUT / 'projections_v2.parquet'
DRIVE_PROJ = OUT_DIR / 'projections_v2.parquet'
MIRROR_EVERY = 10

# Seed local from Drive once if Drive is ahead / local missing (resume across
# a fresh runtime).
if DRIVE_PROJ.exists() and (not PROJ.exists() or DRIVE_PROJ.stat().st_size > PROJ.stat().st_size):
    print(f'copying {DRIVE_PROJ} -> {PROJ} ...', flush=True)
    shutil.copy2(DRIVE_PROJ, PROJ)

cmd = [
    sys.executable, '-u', '-m', 'single_turn_control.run_control',
    '--problems-manifest', str(MANIFEST_PATH),
    '--axis-path', str(AXIS),
    '--model', MODEL,
    '--n-layers', str(N_LAYERS),
    '--output', str(PROJ),
    '--mirror-output', str(DRIVE_PROJ),
    '--mirror-every', str(MIRROR_EVERY),
]
print('CMD:', ' '.join(cmd), flush=True)
print('Quiet while loading Qwen3-32B is normal...', flush=True)

proc = subprocess.Popen(
    cmd, cwd=str(REPO / 'single_turn_control'),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
rc = proc.wait()
print('run_control exit:', rc, flush=True)
assert rc == 0 and PROJ.exists(), PROJ
print('projections ->', PROJ, 'size MB', round(PROJ.stat().st_size / 1e6, 2))
print('Drive mirror:', DRIVE_PROJ)

## Analyze (CPU)

Per-variant + pooled AUROC/CI/permutation at the primary layer (read from
`axis_manifest_32b.json`), plus the full layer sweep.

In [ ]:
import subprocess, sys
from pathlib import Path

REPO = Path('/content/failure_prediction_research')
REPORT_DIR = OUT_DIR / 'report'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, '-u', '-m', 'single_turn_control.analyze',
    '--projections', str(DRIVE_PROJ),
    '--output-dir', str(REPORT_DIR),
]
print('CMD:', ' '.join(cmd), flush=True)
proc = subprocess.Popen(
    cmd, cwd=str(REPO / 'single_turn_control'),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
rc = proc.wait()
print('analyze exit:', rc, flush=True)
assert rc == 0
print('report ->', REPORT_DIR)

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

rep = OUT_DIR / 'report' / 'report.json'
if rep.exists():
    print(json.dumps(json.loads(rep.read_text()), indent=2)[:4000])

plot = OUT_DIR / 'report' / 'layer_sweep.png'
if plot.exists():
    display(Image(filename=str(plot)))
else:
    print('missing', plot)

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

zip_path = OUT_DIR / 'single_turn_control_32b_results.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT_DIR.rglob('*'):
        if p.is_file() and p.suffix.lower() in {'.json', '.png', '.parquet'}:
            z.write(p, p.relative_to(OUT_DIR).as_posix())
print('zip ->', zip_path)
files.download(str(zip_path))